# Model 2 — temperature model with daily offset and robust likelihood

This notebook fits the advanced Bayesian model:

\[
y_i = \log(PR_i)
\]

\[
y_i \sim StudentT(\nu, \mu_i, \sigma)
\]

\[
\mu_i = \alpha + \beta_T x_{T,i} + \delta_{day[i]}
\]

\[
\delta_d \sim Normal(0, \sigma_{day})
\]

The model keeps the physically motivated linear temperature effect, but adds:

1. **Student-t likelihood** — robustness to remaining outliers.
2. **Daily latent offset** — day-specific shift in PR caused by unmodelled slow-varying effects.

The daily offset should not be interpreted as direct soiling measurement. It represents unmodelled day-level variation, such as residual irradiance errors, sensor drift, clouds, soiling-like losses, or other daily operating conditions.

In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az
from cmdstanpy import CmdStanModel

RANDOM_SEED = 42

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

## 1. Load data

Use the moderate-filtering dataset. This should be the same dataset used for Model 1 Gaussian baseline, so that model comparison is fair.

In [ ]:
DATA_PATH = "Bayesian-Data-Analytics-PV-Project/onemin-Ground-2016/2016/ground_model_data_moderate.csv"
df = pd.read_csv(DATA_PATH)
print("Using clean data:", DATA_PATH)

STAN_MODEL_PATH = Path("Bayesian-Data-Analytics-PV-Project/model_2_temperature_daily_offset.stan")
STAN_PRIOR_PATH = Path("Bayesian-Data-Analytics-PV-Project/model_2_temperature_daily_offset_prior_predictive.stan")


print("Using data:", DATA_PATH)
print("Raw data shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

required = ["logPR", "PR", "x_T", "day_id", "T_module_C"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Remove invalid rows just in case.
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=["logPR", "PR", "x_T", "day_id", "T_module_C"]).copy()

# Ensure day_id is integer and starts from 1.
df["day_id"] = pd.factorize(df["day_id"])[0] + 1

print("Model data shape:", df.shape)
print("Number of days:", df["day_id"].nunique())

df[["logPR", "PR", "x_T", "day_id", "T_module_C"]].head()

## 2. Prepare Stan data

The response is \(y=\log(PR)\).

The temperature predictor is:

\[
x_T = \frac{T_m - 25}{10}
\]

so one unit of \(x_T\) corresponds to a \(10^\circ C\) increase in module temperature.

In [ ]:
y = df["logPR"].to_numpy()
x_T = df["x_T"].to_numpy()
day_id = df["day_id"].astype(int).to_numpy()

N = len(df)
J = int(df["day_id"].max())

stan_data = {
    "N": N,
    "J": J,
    "y": y,
    "x_T": x_T,
    "day_id": day_id,
}

prior_data = {
    "N": N,
    "J": J,
    "x_T": x_T,
    "day_id": day_id,
}

print(f"N = {N}")
print(f"J = {J}")
print(f"x_T range = {x_T.min():.3f} ... {x_T.max():.3f}")
print(f"PR range = {df['PR'].min():.3f} ... {df['PR'].max():.3f}")

## 3. EDA motivation for daily offset

Before fitting Model 2, inspect day-to-day variation. If different days have different PR levels, a single global intercept is not enough.

In [ ]:
daily = (
    df.groupby("day_id")
    .agg(
        PR_mean=("PR", "mean"),
        PR_std=("PR", "std"),
        n=("PR", "size"),
        T_mean=("T_module_C", "mean"),
    )
    .reset_index()
)

plt.figure(figsize=(10, 5))
plt.errorbar(daily["day_id"], daily["PR_mean"], yerr=daily["PR_std"], fmt="o-", alpha=0.8)
plt.xlabel("Day ID")
plt.ylabel("Daily mean PR")
plt.title("Daily mean PR with daily standard deviation")
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.scatter(df["day_id"], df["PR"], s=8, alpha=0.35)
plt.axhline(df["PR"].mean(), linestyle="--")
plt.xlabel("Day ID")
plt.ylabel("PR")
plt.title("PR by day — motivation for daily-offset model")
plt.grid(True)
plt.show()

## 4. Prior predictive check

This checks whether the priors generate plausible PR values before seeing the data.

The prior predictive distribution should be wider than the observed data, but it should not be mostly physically absurd.

In [ ]:

prior_model = CmdStanModel(stan_file=str(STAN_PRIOR_PATH))

prior_fit = prior_model.sample(
    data=prior_data,
    fixed_param=True,
    chains=1,
    iter_sampling=1000,
    seed=RANDOM_SEED,
)

alpha_prior = prior_fit.stan_variable("alpha")
beta_T_prior = prior_fit.stan_variable("beta_T")
sigma_prior = prior_fit.stan_variable("sigma")
sigma_day_prior = prior_fit.stan_variable("sigma_day")
nu_prior = prior_fit.stan_variable("nu")
PR_prior = prior_fit.stan_variable("PR_prior")

print("Prior alpha percentiles:", np.percentile(alpha_prior, [5, 50, 95]))
print("Prior beta_T percentiles:", np.percentile(beta_T_prior, [5, 50, 95]))
print("Prior sigma percentiles:", np.percentile(sigma_prior, [5, 50, 95]))
print("Prior sigma_day percentiles:", np.percentile(sigma_day_prior, [5, 50, 95]))
print("Prior nu percentiles:", np.percentile(nu_prior, [5, 50, 95]))
print("Prior predictive PR percentiles:", np.percentile(PR_prior.flatten(), [1, 5, 50, 95, 99]))

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(PR_prior.flatten(), bins=100, density=True, alpha=0.55, label="Prior predictive PR")
plt.hist(df["PR"], bins=80, density=True, alpha=0.55, label="Observed PR")
plt.xlabel("PR")
plt.ylabel("Density")
plt.title("Prior predictive check for measurements")
plt.legend()
plt.grid(True)
plt.show()

fig, axes = plt.subplots(1, 5, figsize=(17, 3.5))
prior_samples = [
    ("alpha", alpha_prior),
    ("beta_T", beta_T_prior),
    ("sigma", sigma_prior),
    ("sigma_day", sigma_day_prior),
    ("nu", nu_prior),
]

for ax, (name, samples) in zip(axes, prior_samples):
    ax.hist(samples, bins=40, alpha=0.8)
    ax.set_title(f"Prior {name}")
    ax.grid(True)

plt.tight_layout()
plt.show()

## 5. Fit Model 2

This is the actual advanced model:

\[
\log(PR_i) \sim StudentT(\nu, \alpha + \beta_T x_{T,i} + \delta_{day[i]}, \sigma)
\]

\[
\delta_d = \sigma_{day} z_d, \quad z_d \sim Normal(0,1)
\]

where daily offsets are centered so that the average daily offset is approximately zero.

In [ ]:


model = CmdStanModel(stan_file=str(STAN_MODEL_PATH))

fit = model.sample(
    data=stan_data,
    chains=4,
    iter_warmup=1000,
    iter_sampling=1000,
    seed=RANDOM_SEED,
    adapt_delta=0.95,
    max_treedepth=12,
)

## 6. Sampling diagnostics

Check:

- no divergent transitions,
- \(\hat{R}\) close to 1,
- reasonable effective sample sizes.

In [ ]:
print(fit.diagnose())

summary = fit.summary()
display(summary.loc[["alpha", "beta_T", "sigma", "sigma_day", "nu"]])

In [ ]:
draws_pd = fit.draws_pd()
div_cols = [c for c in draws_pd.columns if c.endswith("__") and "divergent" in c]
if div_cols:
    n_div = int(draws_pd[div_cols[0]].sum())
else:
    n_div = 0

print("Number of divergences:", n_div)

## 7. Posterior parameter analysis

The main physical parameter is \(\beta_T\), the effect of \(+10^\circ C\) module temperature on \(\log(PR)\).

The daily variability is measured by \(\sigma_{day}\). A non-zero \(\sigma_{day}\) means that day-to-day offsets are relevant.

In [ ]:
alpha = fit.stan_variable("alpha")
beta_T = fit.stan_variable("beta_T")
sigma = fit.stan_variable("sigma")
sigma_day = fit.stan_variable("sigma_day")
nu = fit.stan_variable("nu")
delta_day = fit.stan_variable("delta_day")  # shape: draws x J

effect_10C_pct = 100 * (np.exp(beta_T) - 1)
effect_1C_pct = 100 * (np.exp(beta_T / 10) - 1)

print(f"P(beta_T < 0 | data) = {np.mean(beta_T < 0):.4f}")
print()
print("beta_T posterior percentiles:")
print(np.percentile(beta_T, [5, 50, 95]))
print()
print("Effect per +10 degC [%] percentiles:")
print(np.percentile(effect_10C_pct, [5, 50, 95]))
print()
print("Effect per +1 degC [%] percentiles:")
print(np.percentile(effect_1C_pct, [5, 50, 95]))
print()
print("sigma percentiles:")
print(np.percentile(sigma, [5, 50, 95]))
print()
print("sigma_day percentiles:")
print(np.percentile(sigma_day, [5, 50, 95]))
print()
print("nu percentiles:")
print(np.percentile(nu, [5, 50, 95]))

day_offset_pct = 100 * (np.exp(delta_day) - 1)
print()
print("Daily offset [%] across all days and posterior draws:")
print(np.percentile(day_offset_pct.flatten(), [1, 5, 50, 95, 99]))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].hist(beta_T, bins=50, alpha=0.8)
axes[0].axvline(0, linestyle="--")
axes[0].set_title("Posterior beta_T")
axes[0].set_xlabel("beta_T")

axes[1].hist(effect_10C_pct, bins=50, alpha=0.8)
axes[1].axvline(0, linestyle="--")
axes[1].set_title("Effect of +10°C on PR")
axes[1].set_xlabel("Percent change [%]")

axes[2].hist(sigma, bins=50, alpha=0.8)
axes[2].set_title("Posterior sigma")
axes[2].set_xlabel("sigma")

axes[3].hist(sigma_day, bins=50, alpha=0.8)
axes[3].set_title("Posterior sigma_day")
axes[3].set_xlabel("sigma_day")

for ax in axes:
    ax.grid(True)

plt.tight_layout()
plt.show()

## 8. Daily offsets

The daily offsets show which days are systematically above or below the global temperature trend.

Positive \(\delta_d\): the day has higher PR than expected from temperature alone.

Negative \(\delta_d\): the day has lower PR than expected from temperature alone.

In [ ]:
delta_day_mean = delta_day.mean(axis=0)
delta_day_hdi = az.hdi(delta_day, hdi_prob=0.9)

day_numbers = np.arange(1, J + 1)

plt.figure(figsize=(11, 5))
plt.errorbar(
    day_numbers,
    100 * (np.exp(delta_day_mean) - 1),
    yerr=[
        100 * (np.exp(delta_day_mean) - np.exp(delta_day_hdi[:, 0])),
        100 * (np.exp(delta_day_hdi[:, 1]) - np.exp(delta_day_mean)),
    ],
    fmt="o",
    alpha=0.85,
)
plt.axhline(0, linestyle="--")
plt.xlabel("Day ID")
plt.ylabel("Daily offset on PR scale [%]")
plt.title("Posterior daily offsets")
plt.grid(True)
plt.show()

## 9. Posterior predictive check — marginal PR distribution

This is the same type of check as in Model 1, but it should not be the only model check.

In [ ]:
PR_pred = fit.stan_variable("PR_pred")
y_pred = fit.stan_variable("y_pred")
mu = fit.stan_variable("mu")

plt.figure(figsize=(8, 5))
plt.hist(PR_pred.flatten(), bins=100, density=True, alpha=0.55, label="Posterior predictive PR")
plt.hist(df["PR"], bins=80, density=True, alpha=0.55, label="Observed PR")
plt.xlabel("PR")
plt.ylabel("Density")
plt.title("Posterior predictive check: distribution of PR")
plt.legend()
plt.grid(True)
plt.show()

## 10. Posterior predictive checks by day

This is the key check for Model 2.

Model 2 should reduce systematic daily mean residuals compared with Model 1.

In [ ]:
# Use posterior mean of predicted PR for each observation.
PR_pred_mean = PR_pred.mean(axis=0)

df_check = df.copy()
df_check["PR_pred_mean"] = PR_pred_mean
df_check["residual_PR"] = df_check["PR"] - df_check["PR_pred_mean"]

daily_check = (
    df_check.groupby("day_id")
    .agg(
        observed_mean_PR=("PR", "mean"),
        predicted_mean_PR=("PR_pred_mean", "mean"),
        residual_mean_PR=("residual_PR", "mean"),
        n=("PR", "size"),
    )
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(daily_check["observed_mean_PR"], daily_check["predicted_mean_PR"])
mn = min(daily_check["observed_mean_PR"].min(), daily_check["predicted_mean_PR"].min())
mx = max(daily_check["observed_mean_PR"].max(), daily_check["predicted_mean_PR"].max())
axes[0].plot([mn, mx], [mn, mx], linestyle="--")
axes[0].set_xlabel("Observed daily mean PR")
axes[0].set_ylabel("Predicted daily mean PR")
axes[0].set_title("Daily mean PR: observed vs predicted")
axes[0].grid(True)

axes[1].plot(daily_check["day_id"], daily_check["residual_mean_PR"], "o-")
axes[1].axhline(0, linestyle="--")
axes[1].set_xlabel("Day ID")
axes[1].set_ylabel("Daily mean residual PR")
axes[1].set_title("Model 2 daily mean residuals")
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(df_check["T_module_C"], df_check["residual_PR"], s=10, alpha=0.35)
plt.axhline(0, linestyle="--")
plt.xlabel("Module temperature [°C]")
plt.ylabel("PR residual = observed - predicted")
plt.title("Model 2 residuals vs module temperature")
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.scatter(df_check["day_id"], df_check["residual_PR"], s=10, alpha=0.35)
plt.axhline(0, linestyle="--")
plt.xlabel("Day ID")
plt.ylabel("PR residual = observed - predicted")
plt.title("Model 2 residuals by day")
plt.grid(True)
plt.show()

## 11. Save ArviZ object for model comparison

This saves the fitted model for WAIC / PSIS-LOO comparison.

In [ ]:
idata2 = az.from_cmdstanpy(
    posterior=fit,
    posterior_predictive=["y_pred"],
    observed_data={"y": y},
    log_likelihood=["log_lik"],
)

with open("idata_model2_temperature_daily_offset.pkl", "wb") as f:
    pickle.dump(idata2, f)

print("Saved: idata_model2_temperature_daily_offset.pkl")

## 12. Interpretation template

Use this text after running the notebook and replacing the numerical values with your output:

The advanced model keeps the physically motivated linear temperature effect, but adds a robust Student-t likelihood and a latent daily offset. The posterior for \(\beta_T\) remains negative, which confirms the temperature-related loss in normalized PV performance. The posterior of \(\sigma_{day}\) indicates whether day-to-day variability is relevant. Compared with the Gaussian baseline, this model should better capture systematic daily deviations and remaining outliers.